# Pycavi zero point calibration check

This notebook checks the calibration of swarped image calibrated in the previous step.

- Reads the swarped image and some necessary keywords.
- Read the catalogue used to calibrate the image.
- Produce several plots that can be used to determine the quality of the photometry.
- Saves the resulting plot into a file
- Then fit a 2D Chebyshev to the photometry residuals and plot the corrected magnitudes.



In [ ]:
import os
import sys
import glob
import copy
import logging

import numpy as np
from numpy.polynomial import chebyshev
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

from pykrige.ok import OrdinaryKriging  # Ordinary Kriging from pykrige
from pykrige.uk import UniversalKriging  # Ordinary Kriging from pykrige

from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS

from photutils.aperture import SkyCircularAperture, SkyCircularAnnulus, aperture_photometry


#   from pycavi_astromatic import SExtractor
sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_tools
from pycavi.pycavi_zpcal import extract_calibration, plot_zpcal, get_files_dict


In [ ]:
# set the directory folders
config_dir = os.getcwd()+'/../config/'
config = pycavi_tools.read_pycavi_config(config_dir)

observatory = config['Provenance']['observatory'] # Needed for the atmospheric extinction curve
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
log_dir = config['Directories']['project_logs_dir']

log_file_swarp = log_dir + "/pymeffpp_zpcal_check.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# At this point it is believed that the data reduction process
# should be closely monitored, so we calibrate target/filter sets,
# that is each data partition, one by one.
#
filters = []
reject_observations = []
dir_name = config['Directories']['cal_dir_name']
zpcal_dict = get_files_dict(reduced_std_dir, reduced_science_dir, dir_name, filters, reject_observations)

observation_tags = list(set(zpcal_dict.keys()))
observation_tags.sort()    # Needed to ensure same index in every run
images_to_check = []
for i in range(len(observation_tags)):
    logging.debug("index: {} observation_tag: {}".format(i, observation_tags[i]))
    for j in range(len(zpcal_dict[observation_tags[i]])):
        images_to_check.append(zpcal_dict[observation_tags[i]][j])

for i in range(len(images_to_check)):
    logging.info("index: {} -- file: {}".format(i, images_to_check[i]))



In [ ]:
# Here we loop over the images selected
# See list above to select a data partition to calibrate
# For example, if the list is:
#
# index: 0 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/CP_Pup/Swarp_dir/CP_Pup_r_SDSS.coadd.fits
# index: 1 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_g_SDSS.coadd.fits
# index: 2 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_r_SDSS.coadd.fits
# index: 3 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_NB_659.coadd.fits
# index: 4 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Swarp_dir/SA101_g_SDSS.coadd.fits
# index: 5 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Swarp_dir/SA101_r_SDSS.coadd.fits
# index: 6 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA98/Swarp_dir/SA98_NB_659.coadd.fits
#
# then to calibrate M67_NB_659.coadd.fits set
#
# idx_selected = [3]
#
idx_selected = [3]    # Keep empty to check all files
for i in range(len(images_to_check)):
    
    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue
        
    zpcal = extract_calibration(images_to_check[i])
    plot_zpcal(zpcal)
    
logging.info("-------Done-------")

In [ ]:
#from numpy.polynomial.chebyshev import Chebyshev
from scipy.interpolate import griddata
from numpy.polynomial.chebyshev import chebvander2d
from astropy.stats import sigma_clip
from scipy.stats import norm


# The data is filtered to get obviously bad values and values
# of faint stars which suffer of intrinsic errors which are
# larger than the effect we want to measure.
#
# The call to the filter_data procedure creates the masked arrays
# zpcal.<array>_masked, where array can be any of x, y, z, etc.
#
zpcal.filter_data(dm_min=-0.4, dm_max=0.4, brightest_mag=11.0, faintest_mag=18.5)
zpcal.create_ds9_region()
z_mean, z_median, z_std = sigma_clipped_stats(zpcal.z_masked, sigma=3, maxiters=5)
print("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean, 4), round(z_median, 4), round(z_std, 4)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))

ax1.scatter(zpcal.x_masked, zpcal.z_masked, s=2, color='red')
ax1.axhline(z_mean, color='k', linewidth=1, linestyle='-', label='mean: {}'.format(round(z_mean, 3)))
ax1.axhline(z_mean+z_std, color='k', linewidth=1, linestyle='--', label='mean +- {}'.format(round(z_std, 3)))
ax1.axhline(z_mean-z_std, color='k', linewidth=1, linestyle='--')
ax1.set_title("Magnitude residuals", fontsize=16)
ax1.set_xlabel("RA in pixels", fontsize=14)
ax1.set_ylabel("delta mag", fontsize=14)
ax1.tick_params(axis='x', labelsize=12)
ax1.tick_params(axis='y', labelsize=12)
ax1.legend()

z_val = np.linspace(-0.2, 0.2, 100)
p = norm.pdf(z_val, z_mean, z_std)
ax2.hist(zpcal.z_masked, bins=30, density=True, stacked=True, color='green', alpha=0.7)
plt.plot(z_val, p, 'k', linewidth=2)
ax2.set_title("Distribution of dm", fontsize=16)
ax2.set_xlabel("dm [mag]", fontsize=14)
ax2.set_ylabel("P(dm)", fontsize=14)
ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', labelsize=12)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12,12))

method = "Kriging"
if "Chebyshev" in method:
    # Here I add new code
    # Define the grid size
    n_pts = 50j
    grid_x, grid_y = np.mgrid[1:zpcal.naxis1:n_pts,1:zpcal.naxis2:n_pts]
    
    # we fit a 2D Chebyshev polynomial of degrees deg
    deg = (5, 5)
    zpcal.fit_chebyshev(deg)   

    # we print the polynomials explicitly
    zpcal.print_chebyshev_polynomials()

    # we display the results of the fit as an image
    grid_z = griddata((zpcal.x_masked, zpcal.y_masked), 
                   zpcal.chebyshev_2d_eval(zpcal.x_masked, zpcal.y_masked), 
                  (grid_x, grid_y), method='cubic')
    print("grid_z.T.shape: {}".format(grid_z.T.shape))
    img = ax.imshow(grid_z.T, extent=(1, zpcal.naxis1, 1, zpcal.naxis2), origin='lower', cmap='viridis')
    ax.set_title("Chebyshev fit to magnitude residuals", fontsize=24) 
    save_plot = zpcal.image_path[:-5]+"_cheb_fit_01.png"
    res_plot_str = '_cheb_'
    z_corr = zpcal.z_cheb_corr


elif "Kriging" in method:                       # Not working yet - kernel dies
    # Perform Ordinary Kriging interpolation
    # This is a very expensive process. Data needs to be filtered out.
    # O(N^3+MxN^2) where N: sample data points, M: number of grid points
   
    x_subset = zpcal.x_masked
    y_subset = zpcal.y_masked
    z_subset = zpcal.z_masked
    logging.info("Kriging filtering: number of data points {} ---> {}".format(len(zpcal.x_masked), len(z_subset)))

    # Interpolate onto the grid
    # Here I add new code
    # Define the grid size
    n_pts = 50
    gridx = np.linspace(1, zpcal.naxis1, n_pts)
    gridy = np.linspace(1, zpcal.naxis2, n_pts)

    # Kriging model construction
    UK = UniversalKriging(
        x_subset, y_subset, z_subset,
        variogram_model='gaussian',
        verbose=True,
        enable_plotting=False
    )
    z_kriged, ss = UK.execute('grid', gridx, gridy)
    grid_z = z_kriged
    grid_x, grid_y = np.meshgrid(gridx, gridy)
    print("z_kriged.shape: {} --  grid_x.shape: {}".format(z_kriged.shape, grid_x.shape))
    img = ax.imshow(z_kriged, extent=(1, zpcal.naxis1, 1, zpcal.naxis2), origin='lower', cmap='viridis')
    ax.set_title("Kriging interpolation to magnitude residuals", fontsize=24) 
    save_plot = zpcal.image_path[:-5]+"_krige_fit_01.png"
    res_plot_str = '_krig_'

    # we get the kriging estimates
    corr, _ = UK.execute('points', x_subset+1, y_subset+1)
    z_corr = z_subset - corr
    print("z_corr.shape: {}".format(z_corr.shape))
    
    

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size="5%", pad=0.1)
cbar = plt.colorbar(img, cax=cax)
cbar.set_label('Residual [mag]', fontsize=14)
cbar.ax.tick_params(labelsize=14)

# Define contours and label them
contour_levels_neg = np.arange(-0.20, 0.0, 0.02)
contour_levels_pos = np.arange(0.02, 0.22, 0.02)

contour_neg = ax.contour(grid_x, grid_y, grid_z, cmap='viridis', levels = contour_levels_neg)
contour_pos = ax.contour(grid_x, grid_y, grid_z, cmap='plasma', levels = contour_levels_pos)
zero_contour = ax.contour(grid_x, grid_y, grid_z, colors='white', linestyles='--', linewidths=2, levels = [0])

ax.clabel(contour_neg, inline=True, fontsize=10, fmt='%1.3f')
ax.clabel(contour_pos, inline=True, fontsize=10, fmt='%1.3f')
ax.clabel(zero_contour, inline=True, fontsize=10, fmt='%1.3f')

# Plot the sample of points used to determine the 2D Chebyshev polynomial
XY_tuples = np.array(list(zip(zpcal.x_masked, zpcal.y_masked)))
ax.scatter(XY_tuples[:,0], XY_tuples[:,1], s=2, color='white')
#plt.colorbar(contour, label='Illumination correction')
ax.set_xlabel('RA in pixels', fontsize=16)
ax.set_ylabel('DEC in pixels', fontsize=16)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)


logging.info("Saving chebyshev fit plot to {}".format(save_plot))
plt.savefig(save_plot)

plt.show()



In [ ]:
from scipy.stats import norm

#clipped_corr = np.clip(z_cheb_corr, min_corr, max_corr)


z_mean_u, z_median_u, z_std_u = sigma_clipped_stats(zpcal.z_masked, sigma=3.0, maxiters=5)
print("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean_u, 4), round(z_median_u, 4), round(z_std_u, 4)))

z_mean, z_median, z_std = sigma_clipped_stats(z_corr, sigma=3, maxiters=5)
print("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean, 4), round(z_median, 4), round(z_std, 4)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))

ax1.scatter(zpcal.x_masked, zpcal.z_masked, s=4, color='red', label="mean: {}   stdev: {}".format(round(z_mean_u, 3), round(z_std_u, 3)))
ax1.scatter(zpcal.x_masked, z_corr, color='black', s=4, label='mean: {}  stdev: {}'.format(round(z_mean, 3), round(z_std, 3)))
ax1.axhline(z_mean, color='k', linewidth=1, linestyle='-')
ax1.axhline(z_mean+z_std, color='k', linewidth=1, linestyle='--')
ax1.axhline(z_mean-z_std, color='k', linewidth=1, linestyle='--')
ax1.set_title("Magnitude residuals", fontsize=16)
ax1.set_xlabel("RA in pixels", fontsize=14)
ax1.set_ylabel("delta mag", fontsize=14)
ax1.tick_params(axis='x', labelsize=12)
ax1.tick_params(axis='y', labelsize=12)
ax1.legend()

x_val = np.linspace(-0.2, 0.2, 100)
p = norm.pdf(x_val, z_mean, z_std)
ax2.hist(zpcal.z_masked, bins=30, density=True, stacked=True, color='green', alpha=1.0)
ax2.hist(z_corr, bins=30, density=True, stacked=True, color='blue', alpha=0.7)
plt.plot(x_val, p, 'k', linewidth=2)
ax2.set_title("Distribution of dm", fontsize=16)
ax2.set_xlabel("dm", fontsize=14)
ax2.set_ylabel("P(dm)", fontsize=14)
ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', labelsize=12)

plt.tight_layout()
logging.info("Saving residual plots to {}".format(zpcal.image_path[:-5]+res_plot_str+"res_01.png"))
plt.savefig(zpcal.image_path[:-5]+res_plot_str+"res_01.png")

plt.show()

In [ ]:
r = np.sqrt(zpcal.x_masked**2+zpcal.y_masked**2)
theta = np.degrees(np.arctan2(zpcal.y_masked-zpcal.naxis2/2.0, zpcal.x_masked-zpcal.naxis1/2.0))

# fit a line to the radial dependence
from scipy import stats

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12,5))
fig.suptitle("Residuals of corrected and uncorrected magnitude differences")

slope, intercept, r_value, p_value, std_err = stats.linregress(r, zpcal.z_masked)
ax1.scatter(r, zpcal.z_masked, color='red', s=6, label="uncorrected mags")
xx = np.linspace(0, max(r), 100)
#ax1.plot(xx, slope*xx+intercept, label=f'Fitted line: y = {slope:.3f}x + {intercept:.3f}', color='red')
ax1.plot(xx, slope*xx+intercept, color='red')
ax1.set_xlabel("r [pix]")
ax1.set_ylabel("dm")
               
ax1.scatter(r, z_corr, color='black', s=4, label="corrected mags")
slope, intercept, r_value, p_value, std_err = stats.linregress(r, z_corr)
xx = np.linspace(0, 1.2, 10)
#ax1.plot(xx, slope*xx+intercept, label=f'Fitted line: y = {slope:.3f}x + {intercept:.3f}', color='black')
ax1.plot(xx, slope*xx+intercept, color='black')

ax1.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax1.legend()

ax2.scatter(theta, zpcal.z_masked, color='red', s=6, label="Residual of uncorrected mags")
ax2.scatter(theta, z_corr, color='black', s=4, label="Residual of uncorrected mags")
ax2.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax2.set_xlabel("theta [deg]")
ax2.set_ylabel("dm")
               
ax3.scatter(zpcal.x_masked, zpcal.z_masked, color='red', s=6, label="Residual of uncorrected mags")
ax3.scatter(zpcal.x_masked, z_corr, color='black', s=2, label="Residual of corrected mags")
ax3.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax3.set_xlabel("RA [pix]")
ax3.set_ylabel("dm")

ax4.scatter(zpcal.y_masked, zpcal.z_masked, color='red', s=6, label="Residual of uncorrected mags")
ax4.scatter(zpcal.y_masked, z_corr, color='black', s=2, label="Residual of ucorrected mags")
ax4.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax4.set_xlabel("Dec [pix]")
ax4.set_ylabel("dm")

plt.tight_layout()

logging.info("Saving residual plots to {}".format(zpcal.image_path[:-5]+res_plot_str+"res_02.png"))
plt.savefig(zpcal.image_path[:-5]+res_plot_str+"res_02.png")
               
plt.show()